# Kilonova (KNe) detection efficiency versus the E(B-V) footprint threshold

- Author: Sylvie Dagoret-Campagne
- Creation date: 2026-09-27
- Kernel: conda_py313_opsim53
- Context: SCOC footprint-shrinking study. This notebook extends the `10_DESCMAFDEPTHANDGALCOUNT` series to the
  kilonova (KNe) detection-efficiency metric, `KNePopMetric` - officially part of the "Variables/Transients"
  group of `science_radar_batch` rather than the DESC group, but analyzed here the same way as notebooks
  06 and 07 (same 7 footprint variants, same output conventions).
- This notebook: **09** of the series - `KNePopMetric` (single GW170817-like model and the full Bulla model
  grid) recomputed on each of the 7 dust-threshold footprint variants of the v5.3.6 simulations, with the
  **same injected kilonova population** (fixed random seed) reused for every run, so that differences between
  runs reflect only the cadence/footprint, not Monte Carlo sampling noise.
- Companion notebooks: `01_compareExgalM5withCuts.ipynb`, `03_WeakLensing.ipynb`,
  `06_FOM3x2pts_DESC_TaskForce_demo.ipynb`, `07_SNCounts_DESC_TaskForce_demo.ipynb` (same 7 runs, other
  metrics). Reference notebook for `KNePopMetric` usage: `../06_MAF_DESC_TaskF/04_KNe_DESC_TaskForce_demo.ipynb`
  (baseline only).
- OpSim simulations analyzed (`/Users/dagoret/DATA/OpSim/`):
  - `shrink_fp_dust_0.050_v5.3.6_10yrs.db`
  - `shrink_fp_dust_0.080_v5.3.6_10yrs.db`
  - `shrink_fp_dust_0.120_v5.3.6_10yrs.db`
  - `shrink_fp_dust_0.150_v5.3.6_10yrs.db`
  - `shrink_fp_dust_0.199_v5.3.6_10yrs.db`
  - `baseline_v5.3.6_10yrs.db` (E(B-V) threshold 0.200)
  - `shrink_fp_dust_0.250_v5.3.6_10yrs.db`


## Notebook overview

**What `KNePopMetric` computes** (see `../06_MAF_DESC_TaskF/04_KNe_DESC_TaskForce_demo.ipynb` for the full
description). `generate_kn_pop_slicer` injects `n_events` fake kilonovae at random sky positions, peak times
and luminosity distances into a `UserPointsSlicer` (one "slice point" per event, not a Healpix pixel).
`KNePopMetric` interpolates a Bulla (2019) light-curve template at the actual visits observed at each position
and evaluates several independent detection criteria (`multi_detect`, `ztfrest_simple` (+ `_red`/`_blue`),
`multi_color_detect`, `red_color_detect`, `blue_color_detect`). MAF's reduce-function mechanism splits the
output into seven per-event bundles, each summarized with `lightcurve_summary()`: total detected, total in the
observed footprint, total in the full injected population, and the two **detection efficiencies** (of the
footprint, and of the full population) - the latter is this metric's headline number.

**Choices made in this notebook (all editable in Section 3).**
- `n_events = 100000` for every run (the *official* `science_radar_batch` batch uses 500000; this notebook
  keeps the already-reduced value used in `04_KNe_DESC_TaskForce_demo.ipynb`, since 7 runs x 2 model
  configurations makes the total cost 14x that of the reference notebook).
- **The same injected population for every run**: `generate_kn_pop_slicer(..., seed=42)` is called once per
  run and per model configuration with the *same* `seed`, so the 100000 (sky position, peak time, template,
  distance) tuples are identical across all 7 runs. `KNePopMetric` has no dust-cut parameter of its own (unlike
  `ExgalM5WithCuts`'s `extinction_cut` or `SNNSNMetric`'s `hard_dust_cut`) - the E(B-V) threshold only enters
  through which visits actually exist in each run's database, so fixing the population is what makes the
  comparison across runs a comparison of footprints rather than of independent random draws.
- Two model configurations, exactly as in `04_KNe_DESC_TaskForce_demo.ipynb`: **`single model`** (one
  GW170817-like template, `mej_dyn=0.005, mej_wind=0.050, phi=30, theta=25.8`) and **`all models`** (the entire
  Bulla grid, population-averaged). `d_min=10, d_max=600` Mpc, non-DD visits only
  (`scheduler_note not like 'DD%'`, the convention of this series).
- No explicit "first 10 years" truncation is applied (unlike notebooks 06/07): `generate_kn_pop_slicer`'s
  default `t_start=1, t_end=3652` already confines injected events to the first ~10 years of survey time for
  every run, which is what actually matters here (see the caveats for the one remaining edge effect on the
  11-year baseline file).

**How the figures are built.**
1. Per-run summary tables (single model, all models), all seven criteria.
2. Sky maps (Mollweide/Aitoff scatter, since events are not on a Healpix grid) of detected vs. non-detected
   events for the headline `multi_detect` criterion, single model, for the 7 runs.
3. Detection efficiency (`multi_detect`, and the other criteria) versus the E(B-V) threshold, single vs. all
   models, and the normalized version.
4. Differences between **consecutive** thresholds: since the population is identical across runs, this is done
   **event by event** (which events flip from non-detected to detected, or the reverse) as well as as a
   plain difference of the summary efficiencies.
5. A consistency check: the baseline row should reproduce `04_KNe_DESC_TaskForce_demo.ipynb` almost exactly
   (same `n_events`, same default `seed=42`, same OpSim file).

**Caching.** MAF is run once per run (the `single` and `all` model bundles share one `MetricBundleGroup`, so
each run's database is opened once); the summary tables are cached as one `.csv` per run, and the per-event
`multi_detect` (single model) detection array as one `.npz` per run, both in `data_09_KNE/`.

## Practical note before running

`KNePopMetric` needs the Bulla kilonova model grid, shipped as `.dat` files under
`$RUBIN_SIM_DATA_DIR/maf/bns/`; if `get_kne_filename` returns an empty list below, run
`rs_download_data --dirs maf` first. With `n_events=100000`, 7 runs and 2 model configurations per run, the
first execution of Section 6 makes 14 population runs; expect it to take a while (a single run of this size
takes on the order of a minute in `04_KNe_DESC_TaskForce_demo.ipynb`). The `.csv`/`.npz` caches in
`data_09_KNE/` make every later run instant.

## 1. Imports

In [ ]:
import os
import time
import inspect
from os.path import join, isfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import rubin_sim
import rubin_sim.maf as maf
import rubin_sim.maf.metric_bundles as mb
from rubin_sim.maf.batches.common import lightcurve_summary
from rubin_sim.maf.maf_contrib.kne_metrics import (
    get_kne_filename,
    KNePopMetric,
    generate_kn_pop_slicer,
)

print("rubin_sim version:", rubin_sim.__version__)

## 2. Output directories

In [ ]:
NB_TAG = "KNE"
data_dir = f"data_09_{NB_TAG}"
figs_dir = f"figs_09_{NB_TAG}"
os.makedirs(data_dir, exist_ok=True)
os.makedirs(figs_dir, exist_ok=True)
print("MAF output (data) directory :", os.path.abspath(data_dir))
print("Figures output directory    :", os.path.abspath(figs_dir))

resultsDb = maf.db.ResultsDb(out_dir=data_dir)

## 3. Configuration

In [ ]:
# Directory holding the OpSim databases
OPSIM_DIR = "/Users/dagoret/DATA/OpSim"

# (run_name, E(B-V) threshold that defines the footprint of the run), sorted by increasing threshold
# - identical list to notebooks 01, 03, 06 and 07 of this series
RUNS_INFO = [
    ("shrink_fp_dust_0.050_v5.3.6_10yrs", 0.050),
    ("shrink_fp_dust_0.080_v5.3.6_10yrs", 0.080),
    ("shrink_fp_dust_0.120_v5.3.6_10yrs", 0.120),
    ("shrink_fp_dust_0.150_v5.3.6_10yrs", 0.150),
    ("shrink_fp_dust_0.199_v5.3.6_10yrs", 0.199),
    ("baseline_v5.3.6_10yrs", 0.200),
    ("shrink_fp_dust_0.250_v5.3.6_10yrs", 0.250),
]
RUN_NAMES = [r for r, _ in RUNS_INFO]
DUST_THRESH = dict(RUNS_INFO)
REF_RUN = "baseline_v5.3.6_10yrs"
PAIRS = [(RUN_NAMES[i + 1], RUN_NAMES[i]) for i in range(len(RUN_NAMES) - 1)]


def get_db_path(run_name):
    fname = run_name + ".db"
    candidates = [join(OPSIM_DIR, fname), join(OPSIM_DIR, "sim_baseline", fname)]
    for c in candidates:
        if isfile(c):
            return c
    raise FileNotFoundError(f"OpSim db not found for run {run_name}. Tried: {candidates}")


for run_name, dust in RUNS_INFO:
    try:
        path = get_db_path(run_name)
    except FileNotFoundError:
        path = "NOT FOUND (only a problem if there is no cached result, see Section 6)"
    print(f"{run_name:40s} E(B-V) < {dust:.3f}  ->  {path}")

In [ ]:
N_EVENTS = 100_000  # official batch: 500_000; kept as in 04_KNe_DESC_TaskForce_demo.ipynb for tractability
SEED = 42  # same population for every run - see the Notebook overview above
D_MIN, D_MAX = 10, 600
SQLCONSTRAINT = "scheduler_note not like 'DD%'"

FORCE_RECOMPUTE = False  # True -> ignore the cached results and rerun MAF

# Single, GW170817-like model
single_inj_params = [{"mej_dyn": 0.005, "mej_wind": 0.050, "phi": 30, "theta": 25.8}]
single_filenames = get_kne_filename(single_inj_params)
print(f"'single model': {len(single_filenames)} matching template file(s)")

# Entire model grid
all_filenames = get_kne_filename(None)
print(f"'all models': {len(all_filenames)} template files in the Bulla grid")

CRITERIA = [
    "multi_detect",
    "ztfrest_simple",
    "ztfrest_simple_red",
    "ztfrest_simple_blue",
    "multi_color_detect",
    "red_color_detect",
    "blue_color_detect",
]
HEADLINE = "multi_detect"
HEADLINE_STAT = "Fraction detected of total (mean)"


def config_tag():
    return f"n{N_EVENTS}_seed{SEED}_d{D_MIN}-{D_MAX}"


print("Config tag:", config_tag())

## 4. The MAF classes used

Same as `04_KNe_DESC_TaskForce_demo.ipynb`, Section 3 - kept here for reference.

In [ ]:
print(inspect.getdoc(get_kne_filename))
print("-" * 80)
print(inspect.getdoc(generate_kn_pop_slicer))
print("-" * 80)
print("KNePopMetric reduce functions (one output per detection criterion):")
print(list(KNePopMetric().reduce_funcs.keys()))
print("-" * 80)
print("lightcurve_summary() statistics applied to each of the maps above:")
print([m.name for m in lightcurve_summary()])

## 5. Helper functions

`generate_kn_pop_slicer` returns a `UserPointsSlicer` (scattered injected events on the sky, not a Healpix
grid), so the maps of notebooks 01/03/06/07 are replaced here by Aitoff-projection scatter plots of the
injected events, colored by detection status.

In [ ]:
def run_label(run_name):
    label = f"E(B-V) < {DUST_THRESH[run_name]:.3f}"
    return label + " (baseline)" if run_name.startswith("baseline") else label


def save_fig(fig, name):
    base = join(figs_dir, name)
    fig.savefig(base + ".png", dpi=150, bbox_inches="tight")
    fig.savefig(base + ".pdf", bbox_inches="tight")
    print("Saved:", base + ".png/.pdf")


def summary_table(bgroup, base_metric_name):
    rows = {}
    for c in CRITERIA:
        key = f"{base_metric_name}_{c}"
        rows[c] = bgroup.bundle_dict[key].summary_values
    return pd.DataFrame(rows).T


def summary_csv_path(run_name):
    tag = run_name.replace(".", "_")
    return join(data_dir, f"KNe_summary_{tag}_{config_tag()}.csv")


def events_npz_path(run_name):
    tag = run_name.replace(".", "_")
    return join(data_dir, f"KNe_events_single_{HEADLINE}_{tag}_{config_tag()}.npz")


def ra_dec_to_aitoff(ra, dec):
    # ra, dec in radians, ra in [0, 2pi) -> matplotlib's aitoff axes expect ra in (-pi, pi]
    ra_shifted = np.where(ra > np.pi, ra - 2 * np.pi, ra)
    return -ra_shifted, dec  # flip RA for the usual sky convention (east to the left)


def plot_event_skymaps(events, tag, title):
    fig = plt.figure(figsize=(15, 12))
    for i, (run_name, dust) in enumerate(RUNS_INFO, start=1):
        ra, dec, det = events[run_name]["ra"], events[run_name]["dec"], events[run_name]["detected"]
        x, y = ra_dec_to_aitoff(ra, dec)
        ax = fig.add_subplot(3, 3, i, projection="aitoff")
        ax.scatter(x[~det], y[~det], s=1, color="lightgray", label="not detected")
        ax.scatter(x[det], y[det], s=2, color="crimson", label="detected")
        ax.set_title(f"{run_name}\n{run_label(run_name)}", fontsize=9)
        ax.grid(alpha=0.3)
    fig.suptitle(title, fontsize=16, y=1.02)
    save_fig(fig, f"{tag}_skymaps_allruns")
    plt.show()


def plot_event_transitions(events, tag, title):
    fig = plt.figure(figsize=(16, 10))
    for i, (run_b, run_a) in enumerate(PAIRS, start=1):
        det_b, det_a = events[run_b]["detected"], events[run_a]["detected"]
        ra, dec = events[run_b]["ra"], events[run_b]["dec"]
        x, y = ra_dec_to_aitoff(ra, dec)
        gained, lost, both, neither = det_b & ~det_a, ~det_b & det_a, det_b & det_a, ~det_b & ~det_a
        ax = fig.add_subplot(2, 3, i, projection="aitoff")
        ax.scatter(x[neither], y[neither], s=1, color="lightgray")
        ax.scatter(x[both], y[both], s=1, color="lightgreen")
        ax.scatter(x[gained], y[gained], s=4, color="tab:blue", label="gained")
        ax.scatter(x[lost], y[lost], s=4, color="tab:red", label="lost")
        ax.set_title(f"{run_b}\nminus {run_a}", fontsize=9)
        ax.grid(alpha=0.3)
    fig.suptitle(
        f"{title}: events that flip detection status between consecutive dust thresholds\n"
        "(blue = newly detected, red = no longer detected, green = detected in both, grey = neither)",
        fontsize=14,
        y=1.02,
    )
    save_fig(fig, f"{tag}_transitions_combined")
    plt.show()

## 6. Compute (or reload) the KNe metric for each run

Both model configurations (`single`, `all`) are grouped into a single `MetricBundleGroup` per run, so each
OpSim database is opened only once (7 opens for the whole notebook).

In [ ]:
def build_bundles(run_name):
    single_slicer = generate_kn_pop_slicer(
        n_events=N_EVENTS, n_files=len(single_filenames), d_min=D_MIN, d_max=D_MAX, seed=SEED
    )
    all_slicer = generate_kn_pop_slicer(
        n_events=N_EVENTS, n_files=len(all_filenames), d_min=D_MIN, d_max=D_MAX, seed=SEED
    )
    single_metric = KNePopMetric(
        output_lc=False, file_list=single_filenames, metric_name="KNePopMetric_single"
    )
    all_metric = KNePopMetric(output_lc=False, file_list=all_filenames, metric_name="KNePopMetric_all")
    single_bundle = mb.MetricBundle(
        single_metric,
        single_slicer,
        SQLCONSTRAINT,
        run_name=run_name,
        info_label="single model",
        summary_metrics=lightcurve_summary(),
    )
    all_bundle = mb.MetricBundle(
        all_metric,
        all_slicer,
        SQLCONSTRAINT,
        run_name=run_name,
        info_label="all models",
        summary_metrics=lightcurve_summary(),
    )
    return single_bundle, all_bundle


def load_or_compute(run_name):
    csv_path, npz_path = summary_csv_path(run_name), events_npz_path(run_name)
    if isfile(csv_path) and isfile(npz_path) and not FORCE_RECOMPUTE:
        print(f"[{run_name}] loaded cached results")
        tables = pd.read_csv(csv_path, index_col=[0, 1])
        with np.load(npz_path) as f:
            ev = {"ra": f["ra"], "dec": f["dec"], "detected": f["detected"]}
        return tables, ev

    dbpath = get_db_path(run_name)
    print(f"[{run_name}] running KNePopMetric (single + all models, n_events={N_EVENTS}) on {dbpath}")
    single_bundle, all_bundle = build_bundles(run_name)
    bd = mb.make_bundles_dict_from_list([single_bundle, all_bundle])
    group = mb.MetricBundleGroup(bd, dbpath, out_dir=data_dir, results_db=resultsDb)
    t0 = time.time()
    group.run_all()
    print(f"  [{run_name}] done in {time.time() - t0:.1f} s")

    single_table = summary_table(group, "KNePopMetric_single")
    all_table = summary_table(group, "KNePopMetric_all")
    tables = pd.concat({"single": single_table, "all": all_table}, names=["model", "criterion"])
    tables.to_csv(csv_path)

    hb = group.bundle_dict[f"KNePopMetric_single_{HEADLINE}"]
    ra = hb.slicer.slice_points["ra"]
    dec = hb.slicer.slice_points["dec"]
    detected = np.ma.filled(hb.metric_values, 0).astype(bool) & ~np.ma.getmaskarray(hb.metric_values)
    np.savez_compressed(npz_path, ra=ra, dec=dec, detected=detected)

    return tables, {"ra": ra, "dec": dec, "detected": detected}

In [ ]:
t0 = time.time()
results = {run_name: load_or_compute(run_name) for run_name, _ in RUNS_INFO}
print(f"Done in {time.time() - t0:.1f} s")

tables = {r: results[r][0] for r in RUN_NAMES}
events = {r: results[r][1] for r in RUN_NAMES}

## 7. Per-run summary tables

`single` = GW170817-like model; `all` = full Bulla grid (population-averaged).

In [ ]:
for run_name, dust in RUNS_INFO:
    print(f"--- {run_name} ({run_label(run_name)}) ---")
    display(tables[run_name].round(4))

## 8. Sky maps of the injected events, single model, `multi_detect`

Same 100000 injected sky positions in every panel (fixed seed); only which ones are detected changes with the
run's actual cadence and footprint.

In [ ]:
plot_event_skymaps(
    events, "KNe_single_multidetect", "KNePopMetric (single model): multi_detect, injected events"
)

## 9. Detection efficiency versus the E(B-V) threshold

`Fraction detected of total (mean)` for the `multi_detect` criterion, single vs. all models.

In [ ]:
dust_values = [d for _, d in RUNS_INFO]
i_ref = RUN_NAMES.index(REF_RUN)


def eff(run_name, model, criterion=HEADLINE, stat=HEADLINE_STAT):
    return tables[run_name].loc[(model, criterion), stat]


single_eff = np.array([eff(r, "single") for r in RUN_NAMES])
all_eff = np.array([eff(r, "all") for r in RUN_NAMES])

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.plot(dust_values, single_eff * 100, "o-", color="steelblue", label="single model (GW170817-like)")
ax.plot(dust_values, all_eff * 100, "s-", color="indianred", label="all models (full Bulla grid)")
ax.axvline(dust_values[i_ref], color="gray", ls="--", linewidth=0.8, label="baseline threshold")
ax.set_xlabel("E(B-V) threshold that defines the footprint of the run")
ax.set_ylabel("KNe detection efficiency [%]\n(multi_detect, fraction of injected events)")
ax.set_title("KNePopMetric: detection efficiency versus dust threshold")
ax.grid(alpha=0.3)
ax.legend(fontsize=9)
fig.tight_layout()
save_fig(fig, "KNe_efficiency_vs_dust_threshold")
plt.show()

### 9.1 Same plot, normalized to the baseline efficiency

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.5))
ax.plot(dust_values, single_eff / single_eff[i_ref], "o-", color="steelblue", label="single model")
ax.plot(dust_values, all_eff / all_eff[i_ref], "s-", color="indianred", label="all models")
ax.axhline(1.0, color="gray", ls="--", linewidth=0.8)
ax.axvline(dust_values[i_ref], color="gray", ls="--", linewidth=0.8, label="baseline")
ax.set_xlabel("E(B-V) threshold that defines the footprint of the run")
ax.set_ylabel("Detection efficiency / baseline")
ax.set_title("Same quantity, normalized to the baseline run")
ax.grid(alpha=0.3)
ax.legend(fontsize=9)
fig.tight_layout()
save_fig(fig, "KNe_efficiency_vs_dust_threshold_norm")
plt.show()

### 9.2 All seven criteria, single model vs. all models

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
x = np.arange(len(CRITERIA))
width = 0.35
single_all_crit = [eff(REF_RUN, "single", c) * 100 for c in CRITERIA]
all_all_crit = [eff(REF_RUN, "all", c) * 100 for c in CRITERIA]
ax.bar(x - width / 2, single_all_crit, width, label="single model", color="steelblue")
ax.bar(x + width / 2, all_all_crit, width, label="all models", color="indianred")
ax.set_xticks(x)
ax.set_xticklabels(CRITERIA, rotation=30, ha="right")
ax.set_ylabel("Detection efficiency [%]")
ax.set_title(f"KNe detection efficiency by criterion - {REF_RUN} (baseline)")
ax.legend()
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
save_fig(fig, "KNe_efficiency_by_criterion_baseline")
plt.show()

## 10. Differences between consecutive dust thresholds

**Efficiency differences** (summary level, all criteria) and, since the injected population is identical
across runs, **event-by-event transitions** for the headline `multi_detect` criterion (single model): which
events go from not detected to detected, or the reverse, as the threshold increases.

In [ ]:
rows = []
for run_b, run_a in PAIRS:
    row = {"pair (E(B-V) thresholds)": f"{DUST_THRESH[run_b]:.3f} - {DUST_THRESH[run_a]:.3f}"}
    for model in ("single", "all"):
        for c in CRITERIA:
            row[f"{model}:{c}"] = eff(run_b, model, c) - eff(run_a, model, c)
    rows.append(row)
diff_stats = pd.DataFrame(rows).set_index("pair (E(B-V) thresholds)")
diff_stats.to_csv(join(data_dir, "KNe_efficiency_diff_summary.csv"))
print("Saved:", join(data_dir, "KNe_efficiency_diff_summary.csv"))
display(diff_stats[[f"single:{HEADLINE}", f"all:{HEADLINE}"]].round(4))

In [ ]:
plot_event_transitions(events, "KNe_single_multidetect", "KNePopMetric (single model)")

## 11. Consistency check against notebook 04

The baseline row uses the same `n_events`, the same default `seed=42` and the same OpSim file as
`../06_MAF_DESC_TaskF/04_KNe_DESC_TaskForce_demo.ipynb`, so it should reproduce that notebook's headline
numbers exactly (up to the small `night <= 10*365.25+0.5` vs. `t_end=3652` difference discussed in the
caveats below).

In [ ]:
print(f"single model, multi_detect, baseline: {eff(REF_RUN, 'single'):.1%}")
print(f"all models,   multi_detect, baseline: {eff(REF_RUN, 'all'):.1%}")

## Caveats

- These are **detection efficiencies**, not expected event counts: turning a fraction into "expected KNe
  detections per year" requires an actual binary-neutron-star/NS-BH merger volumetric rate density, which is
  not part of this metric.
- `n_events = 100000` (not the official 500000): efficiency estimates carry Monte Carlo (binomial) sampling
  noise of order `1/sqrt(n_events) ~ 0.3%`; because the **same** population is reused for every run (fixed
  `seed`), this noise is common to all runs and mostly cancels in the differences/ratios plotted in Sections
  9-10, but still affects the absolute efficiency of Section 7/9 and the consistency check of Section 11.
- No explicit "first 10 years" truncation is applied to the SQL query (unlike notebooks 06/07): the injected
  events already stop at `t_end=3652` days (~10 years) by default. Because each event's detection window
  extends up to ~30 days past peak, the handful of events peaking closest to day 3652 can still pick up a few
  extra visits in the baseline file (which has an 11th year available) that the 10-year files do not have -
  a small, one-sided edge effect, not corrected here (reduce `t_end` in `generate_kn_pop_slicer` by ~30-60
  days for a perfectly symmetric comparison if this matters for your use case).
- The `"all models"` run averages uniformly over the *entire* Bulla template grid (all ejecta masses, opening
  angles and viewing angles with equal weight), which is not necessarily how real kilonovae are distributed in
  nature; the `"single model"` (GW170817-like) run instead isolates one specific, observationally anchored
  scenario.
- `KNePopMetric` has no dust-cut parameter analogous to `extinction_cut`/`hard_dust_cut`: extinction is applied
  internally via `DustMap` to dim the simulated light curves, and the only way the E(B-V) footprint threshold
  enters this notebook is through which visits exist in each run - unlike notebooks 01/03/06/07 there is no
  `EBV_CUT_MODE` here.
- A more sophisticated companion metric, `PrestoColorKNePopMetric`, additionally models photometric
  *classification* probability (not just detection); it is not covered by this notebook, mirroring the
  scope of `04_KNe_DESC_TaskForce_demo.ipynb`.
- `0.199` and `baseline (0.200)` have almost the same nominal footprint: the difference between their
  efficiencies gives the scale of the run-to-run (scheduler) fluctuations, to compare against the change seen
  between the other, more widely separated thresholds.

## References
- Bulla, M. 2019, MNRAS 489, 5037, "POSSIS: predicting spectra, light curves and polarization for
  multi-dimensional models of supernovae and kilonovae" - defines the kilonova template grid used here.
- Andreoni, I., Coughlin, M. W. et al. 2021, ApJ 918, 63, "Fast-transient Searches in Real Time with ZTFReST"
  - origin of the `ztfrest_simple` rise/fade-rate detection criterion.
- Andrade, C. et al. 2025, PASP, "The Effect of Vera C. Rubin Observatory Cadence Selections on Kilonova
  Detectability" (arXiv:2502.14124) - a published cadence study using exactly `KNePopMetric`.
- Bianco, F. B. et al. 2022, ApJS 258, 1 - SCOC context.
- `rubin_sim.maf` documentation: https://rubin-sim.lsst.io/maf.html
- `rubin_sim` source: https://github.com/lsst/rubin_sim (`rubin_sim/maf/maf_contrib/kne_metrics.py`,
  `rubin_sim/maf/batches/science_radar_batch.py`)
- `../06_MAF_DESC_TaskF/04_KNe_DESC_TaskForce_demo.ipynb` - the `KNePopMetric` chain, baseline only (this
  notebook's direct model).
- `01_compareExgalM5withCuts.ipynb`, `03_WeakLensing.ipynb`, `06_FOM3x2pts_DESC_TaskForce_demo.ipynb`,
  `07_SNCounts_DESC_TaskForce_demo.ipynb` - the E(B-V) footprint-threshold comparison for the other metrics of
  this series.
- `shrink_fp_dust_*.db` simulations: https://s3df.slac.stanford.edu/data/rubin/sim-data/sims_featureScheduler_runs5.3/shrink_fp/